Updated parcellation script: Re-coded to support Schaefer and MIST atlas parcellation options.

[Runtime: Trivial time/computation requirements, compared to the preceding processing stages]

----------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, csv, json, math, glob, re, time
import subprocess
from pathlib import Path
import shutil
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
### LOAD FREESURFER:
FREESURFER_HOME = config['freesurfer']['home']
FREESURFER_LICENSE = Path(config['freesurfer'].get('license'))
FREESURFER_SUBJECTS_DIR = config['freesurfer']['subjects_dir']
os.environ['FREESURFER_HOME'] = str(FREESURFER_HOME)
os.environ['SUBJECTS_DIR']    = str(FREESURFER_SUBJECTS_DIR)
os.environ['PATH'] = f"{str(FREESURFER_HOME)}/bin:" + os.environ.get('PATH', '')
if FREESURFER_LICENSE.is_file():
    os.environ['FS_LICENSE'] = str(FREESURFER_LICENSE)
else:
    raise RuntimeError("FreeSurfer license not found: check that 'license.txt' exists in home FreeSurfer directory."
    "in FreeSurfer $HOME directory and that a valid path is set in 'config.yaml' file.")
subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
# __________________________________________________________________________________________________________
### LOAD PARAMETERS:

# General parameters:

HARD_STOP = config['hard_errors']
RANDOM_SEED = config['random_seed']

SUBSET = config['subset']

# Optional filtering params:
FILTER_SUBJECT_IDS = config["filter"].get("subject_IDs", [])
FILTER_SESSION_IDS = config["filter"].get("session_IDs", [])
FILTER_GROUP_IDS   = config["filter"].get("group_IDs", [])

OVERWRITE = config['overwrite_parcellations']


# Processing parameters:

APPLY_GM_CLIP = config['apply_GM_clip']
EPS_ZERO_SERIES = config['zero_threshold']


# Time-series type selection (mean vs norm) from config.yaml
TIMESERIES_TYPE = str(config['parcellation'].get('timeseries_type', "")).strip().lower()
if TIMESERIES_TYPE not in ("mean", "norm"):
    msg = (
        f"Invalid config['parcellation']['timeseries_type'] = '{TIMESERIES_TYPE}'. "
        "Must be 'mean' or 'norm'.")
    print(f"[CONFIG ERROR] {msg}")
    raise ValueError(msg)


# __________________________________________________________________________________________________________
### ATLAS SELECTION (family / n_rois / networks) FROM YAML:

PARCELLATION        = config['parcellation']                # <-- expects keys: 'atlas', 'n_rois', (optional / configuration-dependent) 'network_scale'
ATLAS_FAMILY        = str(PARCELLATION['atlas']).strip()    # <-- "Craddock" | "Schaefer" | "MIST"
AF_LOWER            = ATLAS_FAMILY.lower()
ATLAS_N_ROIS        = int(PARCELLATION['n_rois'])
ATLAS_NETWORK_SCALE = PARCELLATION.get('network_scale', None)  # <-- used only for Schaefer atlas

atlases_block = config['atlases']

if AF_LOWER == "craddock":
    cr_cfg = atlases_block['Craddock']
    CRADDOCK_DIR      = Path(cr_cfg['craddock_dir'])
    CANONICAL_MNI     = cr_cfg.get('canonical_mni', 'MNI152NLin2009cAsym')
    CANONICAL_RES_MM  = int(cr_cfg.get('canonical_res', 4))
    allowed_cr_counts = [int(x) for x in cr_cfg.get('allowed_num_ROIs', [50, 100, 200])]

    if ATLAS_N_ROIS not in allowed_cr_counts:
        raise ValueError(
            f"Craddock: requested n_rois={ATLAS_N_ROIS}, but allowed_num_ROIs={allowed_cr_counts}")

    # Craddock does not use network_scale:
    if ATLAS_NETWORK_SCALE not in (None, "", "None"):
        print("[INFO] Craddock atlas does not use 'network_scale'; ignoring provided value.")
    ATLAS_NETWORK_SCALE = None

elif AF_LOWER == "schaefer":
    sch_cfg = atlases_block['Schaefer']

    # For Schaefer, network_scale is required (7 or 17):
    if ATLAS_NETWORK_SCALE in (None, "", "None"):
        raise ValueError(
            "Schaefer atlas requires 'parcellation.network_scale' (e.g., 7 or 17).")
    ATLAS_NETWORK_SCALE = int(ATLAS_NETWORK_SCALE)

    allowed_scales = sch_cfg.get('allowed_scales', {})  # <-- e.g. {100: [7, 17], ...}
    if ATLAS_N_ROIS not in allowed_scales:
        raise ValueError(
            f"Schaefer: requested n_rois={ATLAS_N_ROIS}, but allowed_scales keys are {list(allowed_scales.keys())}")

    allowed_networks = [int(x) for x in allowed_scales[ATLAS_N_ROIS]]
    if allowed_networks and (ATLAS_NETWORK_SCALE not in allowed_networks):
        raise ValueError(
            f"Schaefer: requested n_rois={ATLAS_N_ROIS} with network_scale={ATLAS_NETWORK_SCALE}, "
            f"but allowed_scales[{ATLAS_N_ROIS}] = {allowed_networks}")

elif AF_LOWER == "mist":
    mist_cfg = atlases_block['MIST']
    allowed_rois = [int(x) for x in mist_cfg.get('allowed_num_ROIs', [])]
    if allowed_rois and (ATLAS_N_ROIS not in allowed_rois):
        raise ValueError(
            f"MIST: requested n_rois={ATLAS_N_ROIS}, but allowed_num_ROIs={allowed_rois}")

    # MIST uses only n_rois; ignore any network_scale:
    if ATLAS_NETWORK_SCALE not in (None, "", "None"):
        print("[INFO] MIST atlas ignores 'network_scale'; using n_rois only.")
    ATLAS_NETWORK_SCALE = None

else:
    raise ValueError(f"Unsupported atlas family: {ATLAS_FAMILY}")

# Derived “tag” and EPI filename for this run:
if AF_LOWER == "craddock":
    ATLAS_TAG = f"craddock-{ATLAS_N_ROIS:03d}"
elif AF_LOWER == "schaefer":
    ATLAS_TAG = f"schaefer-{ATLAS_N_ROIS:03d}p-{ATLAS_NETWORK_SCALE}net"
elif AF_LOWER == "mist":
    ATLAS_TAG = f"mist-{ATLAS_N_ROIS}"
else:
    ATLAS_TAG = AF_LOWER

# This MUST match the filename written by the alignment script (#06):
ATLAS_EPI_FILENAME = f"atlas_{ATLAS_TAG}_on_EPI.nii.gz"

print(
    f"[ATLAS] family={ATLAS_FAMILY} | n_rois={ATLAS_N_ROIS} | "
    f"network_scale={ATLAS_NETWORK_SCALE} | tag='{ATLAS_TAG}' | "
    f"EPI filename='{ATLAS_EPI_FILENAME}'")


# __________________________________________________________________________________________________________
### SET FILEPATHS:

### INPUT:
BASE_DIRECTORY = Path(config['root_output_directory'])

RUN_MANIFEST_PATH   = BASE_DIRECTORY / 'subject_manifest.csv'
fMRI_PARAMETERS_PATH = BASE_DIRECTORY / 'fMRI_manifest.csv'

DATA_PATH = Path(BASE_DIRECTORY) / config['denoising_output_dir']

BRAINMAP_DIR = Path(BASE_DIRECTORY) / config['alignment_output_dir']

BRAINMASK_PATH = Path(BASE_DIRECTORY) / config['BBR_output_dir']


### OUTPUT:

PARCELLATION_OUTPUT_DIR = Path(BASE_DIRECTORY) / config['parcellation_output_dir']
PARCELLATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# __________________________________________________________________________________________________________
### INITIALIZATION:

# Load DataFrames:
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs    = pd.read_csv(fMRI_PARAMETERS_PATH)

In [ ]:
# =====================================================================
# FILTERING (group_ID → session_ID → subject_ID) + DIAGNOSTIC SUBSETTING
# =====================================================================

# ---- FILTERING (if enabled) ----
any_filters_active = bool(FILTER_GROUP_IDS or FILTER_SESSION_IDS or FILTER_SUBJECT_IDS)
if any_filters_active:
    print("\n[FILTER] Applying YAML-defined filters to fMRI_runs...")
    print(f"[FILTER] Starting with {len(fMRI_runs):,} rows.")
    # 1) Filter by group_IDs (substrings, case-insensitive):
    if FILTER_GROUP_IDS:
        n_before = len(fMRI_runs)
        pattern = "|".join(re.escape(val) for val in FILTER_GROUP_IDS)
        mask = fMRI_runs["group_ID"].astype(str).str.contains(pattern, case=False, na=False)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] group_IDs {FILTER_GROUP_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 2) Filter by session_IDs (exact matches):
    if FILTER_SESSION_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["session_ID"].astype(str).isin(FILTER_SESSION_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] session_IDs {FILTER_SESSION_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 3) Filter by subject_IDs (exact matches):
    if FILTER_SUBJECT_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["subject_ID"].astype(str).isin(FILTER_SUBJECT_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] subject_IDs {FILTER_SUBJECT_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    print(f"[FILTER] Final row count after all filters: {len(fMRI_runs):,} rows.\n")

# ---- DIAGNOSTIC SUBSETTING (if enabled) ----
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    fMRI_runs = fMRI_runs.head(SUBSET).copy()

if any_filters_active or SUBSET:
    fMRI_runs = fMRI_runs.reset_index(drop=True)
    display(fMRI_runs)

------------

"Pre-flight" checks:

In [ ]:
# __________________________________________________________________________________________________________
### Build + audit filetargets_df from fMRI_runs (using atlas family + EPI-space files)

# Reconstruct the expected EPI-space atlas filename (based on the same convention used in script #06):
family_lower = ATLAS_FAMILY.lower()

# Use 3-digit zero-padded ROI count for EPI-space filenames (e.g., 050, 100, 200):
roi_str = f"{ATLAS_N_ROIS:03d}"

if family_lower == "craddock":
    # e.g., atlas_craddock-050_on_EPI.nii.gz
    ATLAS_EPI_FILENAME = f"atlas_craddock-{roi_str}_on_EPI.nii.gz"

elif family_lower == "schaefer":
    if ATLAS_NETWORK_SCALE is None:
        raise RuntimeError(
            "Parcellation configured with atlas_family='Schaefer' but "
            "'network_scale' is not set in config['parcellation'].")
    # e.g., atlas_schaefer-100p-7net_on_EPI.nii.gz (and atlas_schaefer-050p-7net_on_EPI.nii.gz if n_rois < 100)
    ATLAS_EPI_FILENAME = f"atlas_schaefer-{roi_str}p-{ATLAS_NETWORK_SCALE}net_on_EPI.nii.gz"

elif family_lower == "mist":
    # e.g., atlas_mist-020_on_EPI.nii.gz
    ATLAS_EPI_FILENAME = f"atlas_mist-{roi_str}_on_EPI.nii.gz"

else:
    raise ValueError(f"Unsupported atlas family for parcellation: {ATLAS_FAMILY!r}")

print(f"[PARC] Atlas family={ATLAS_FAMILY}, n_rois={ATLAS_N_ROIS}, "
      f"network_scale={ATLAS_NETWORK_SCALE} -> EPI filename='{ATLAS_EPI_FILENAME}'")

# -----------------------------------------------------------------------
# Verify we have the columns we need from 'fMRI_runs':
required_cols = ["subject_ID", "session_ID"]
for col in required_cols:
    if col not in fMRI_runs.columns:
        raise RuntimeError(f"fMRI_runs is missing required column: '{col}'")

df_ready = fMRI_runs.copy()
df_ready["subject_ID"] = df_ready["subject_ID"].astype(str)
df_ready["session_ID"] = df_ready["session_ID"].astype(str)

# Our canonical identifier 'prefix' is always '{<subject_ID>_<session_ID>}':
df_ready["prefix"] = df_ready["subject_ID"] + "_" + df_ready["session_ID"]

records  = []
problems = []

for row in df_ready.itertuples(index=False):
    subj   = str(row.subject_ID)
    sess   = str(row.session_ID)
    prefix = f"{subj}_{sess}"

    fmri_path      = DATA_PATH      / prefix / "bold_denoised.nii.gz"
    atlas_path     = BRAINMAP_DIR   / prefix / ATLAS_EPI_FILENAME
    brainmask_path = BRAINMASK_PATH / prefix / "brainmask_epi.nii.gz"
    gm_mask_path   = BRAINMASK_PATH / prefix / "gm_epi.nii.gz"

    missing = []
    if not fmri_path.exists():      missing.append("bold_denoised.nii.gz")
    if not atlas_path.exists():     missing.append(ATLAS_EPI_FILENAME)
    if not brainmask_path.exists(): missing.append("brainmask_epi.nii.gz")
    if not gm_mask_path.exists():   missing.append("gm_epi.nii.gz")

    if missing:
        problems.append({
            "subject_ID": subj,
            "session_ID": sess,
            "prefix": prefix,
            "issue": "missing_inputs",
            "detail": ", ".join(missing)})

    records.append({
        "subject_ID": subj,
        "session_ID": sess,
        "prefix": prefix,
        "fMRI_path": str(fmri_path),
        "brainmap_path": str(atlas_path),
        "brainmask_path": str(brainmask_path),
        "GM_mask_path": str(gm_mask_path),
        "atlas_family": ATLAS_FAMILY,
        "atlas_n_rois": ATLAS_N_ROIS,
        "atlas_network_scale": ATLAS_NETWORK_SCALE,
        "atlas_epi_filename": ATLAS_EPI_FILENAME,
        "missing": ", ".join(missing) if missing else ""})

filetargets_df = (
    pd.DataFrame(records)
      .sort_values(["subject_ID", "session_ID"])
      .reset_index(drop=True))

print(f"[FILES] Built filetargets_df with {len(filetargets_df)} rows "
      f"for atlas family={ATLAS_FAMILY} (EPI filename pattern='{ATLAS_EPI_FILENAME}').")

problems_df = None
if problems:
    problems_df = (
        pd.DataFrame(problems)
          .sort_values(["subject_ID", "session_ID"])
          .reset_index(drop=True))
    print(f"[FILES] WARNING: {len(problems_df)} rows have missing inputs.")
    display(problems_df)

    if HARD_STOP:
        raise RuntimeError(
            "Some rows have missing required inputs. See 'problems_df' above. "
            "Set hard_errors: False in config.yaml to drop problematic rows instead.")
    else:
        print("[FILES] HARD_STOP=False → dropping problematic rows from filetargets_df.")
        bad_prefixes = set(problems_df["prefix"])
        filetargets_df = filetargets_df[~filetargets_df["prefix"].isin(bad_prefixes)].reset_index(drop=True)
        df_ready       = df_ready[~df_ready["prefix"].isin(bad_prefixes)].reset_index(drop=True)

with pd.option_context("display.max_rows", 10, "display.max_colwidth", 120):
    display(filetargets_df.head())

Perform actual parcellation:

In [ ]:
### Extract per-ROI time-series (single type: MEAN or NORM) with atlas-aware filenames + JSON sidecars:

def _affine_close(a, b, tol=1e-4):
    return np.allclose(a, b, atol=tol, rtol=0)

def _zscore_sample(X, axis=0):
    X = np.asarray(X, dtype=np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        mu = np.nanmean(X, axis=axis, keepdims=True)
        sd = np.nanstd(X,  axis=axis, ddof=1, keepdims=True)
        bad = (~np.isfinite(sd)) | (sd == 0)
        sd[bad] = 1.0
        Z = (X - mu) / sd
        Z[~np.isfinite(Z)] = 0.0
    return Z

def _is_integer_like(arr, atol=1e-6):
    a = np.asanyarray(arr)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return True
    return np.max(np.abs(a - np.rint(a))) <= atol

# __________________________________________________________________________________________________________
# Main execution:

for r in filetargets_df.itertuples(index=False):
    prefix = str(r.prefix)
    f_path = Path(r.fMRI_path)
    a_path = Path(r.brainmap_path)
    bm_p   = Path(r.brainmask_path)
    gm_p   = Path(r.GM_mask_path)

    # Input checks:
    missing = [p for p in [f_path, a_path, bm_p, gm_p] if not p or not p.exists()]
    if missing:
        print(f"[SKIP] {prefix}: missing inputs: " + ", ".join(str(m) for m in missing))
        continue

    # Output directory + filenames (new schema; no GMclip, single timeseries type):
    out_dir = PARCELLATION_OUTPUT_DIR / prefix
    out_dir.mkdir(parents=True, exist_ok=True)

    # Build base filename stem:
    # '<prefix>_<timeseries_type>_<atlas>_<num_ROIs>[_<network_scale>]'
    ts_token      = TIMESERIES_TYPE.upper()    # <-- must be either 'mean' or 'norm'
    atlas_token   = ATLAS_FAMILY               # <-- must be "Craddock", "Schaefer", or "MIST"
    n_rois_token  = str(ATLAS_N_ROIS)

    name_parts = [prefix, ts_token, atlas_token, n_rois_token]
    if AF_LOWER == "schaefer" and (ATLAS_NETWORK_SCALE is not None):
        name_parts.append("net"+str(ATLAS_NETWORK_SCALE))

    base_stem = "_".join(name_parts)

    csv_out       = out_dir / f"{base_stem}.csv"
    json_sidecar  = out_dir / f"{base_stem}_info.json"

    if (not OVERWRITE) and csv_out.exists() and json_sidecar.exists():
        print(f"[SKIP] {prefix}: outputs already exist and OVERWRITE=False.")
        continue

    # Load data:
    img = nib.load(str(f_path))
    dat = img.get_fdata(dtype=np.float32)  # <-- dimensionality = (X,Y,Z,T)
    if dat.ndim != 4:
        print(f"[SKIP] {prefix}: fMRI not 4D.")
        continue
    X, Y, Z, T = dat.shape

    atlas_img = nib.load(str(a_path))
    atlas_dat = atlas_img.get_fdata()
    brain_img = nib.load(str(bm_p))
    brain_m   = (brain_img.get_fdata() > 0.5)
    gm_img    = nib.load(str(gm_p))
    gm_m      = (gm_img.get_fdata() > 0.5)

    # Lattice sanity-check:
    ok_aff = (
        _affine_close(img.affine, atlas_img.affine) and
        _affine_close(img.affine, brain_img.affine) and
        _affine_close(img.affine, gm_img.affine))
    ok_xyz = (
        atlas_img.shape[:3] == (X, Y, Z) and
        brain_img.shape[:3] == (X, Y, Z) and
        gm_img.shape[:3]    == (X, Y, Z))
    if not (ok_aff and ok_xyz):
        print(f"[SKIP] {prefix}: lattice mismatch (affine or shape).")
        continue

    # ROI labels (integer-like, >0):
    if not _is_integer_like(atlas_dat):
        print(f"[WARN] {prefix}: atlas not strictly integer-like; rounding.")
        atlas_dat = np.rint(atlas_dat)
    atlas_lab = atlas_dat.astype(np.int32, copy=False)

    labels = np.unique(atlas_lab[atlas_lab > 0])
    if labels.size == 0:
        print(f"[SKIP] {prefix}: no atlas labels > 0.")
        continue

    print(f"[EXTRACT] {prefix}: T={T} | ROIs={labels.size} | GM-clip={APPLY_GM_CLIP} | atlas_tag={ATLAS_TAG}")

    # Support mask (Brain ∧ (optional) GM):
    support = brain_m
    if APPLY_GM_CLIP:
        support = support & gm_m
    support = support.reshape(-1)

    # Flatten for efficient indexing:
    data_2d   = dat.reshape(-1, T)             # <-- (V x T)
    labs_flat = atlas_lab.reshape(-1)          # <-- (V,)

    # Build per-ROI index lists:
    roi_indices = {}
    roi_counts  = {}
    for lab in labels:
        lab_int = int(lab)
        idx = np.where((labs_flat == lab_int) & support)[0]
        roi_indices[lab_int] = idx
        roi_counts[lab_int]  = int(idx.size)

    # Extract MEAN per ROI (T x N):
    N = labels.size
    mean_ts = np.zeros((T, N), dtype=np.float32)
    for j, lab in enumerate(labels):
        lab_int = int(lab)
        idx = roi_indices[lab_int]
        if idx.size == 0:
            continue
        mean_ts[:, j] = np.nanmean(data_2d[idx, :], axis=0, dtype=np.float64)

    # Z-score version (sample-z per ROI) – still computed for diagnostics & possible output:
    z_ts = _zscore_sample(mean_ts, axis=0)     # <-- (T x N)

    # Diagnostics: empty-by-count & “effectively zero”:
    empty_by_count   = [int(lab) for lab in labels if roi_counts[int(lab)] == 0]
    absmax           = np.max(np.abs(mean_ts), axis=0)               # (N,)
    zero_like_mask   = absmax < EPS_ZERO_SERIES
    n_zero_like      = int(np.sum(zero_like_mask))
    zero_like_labels = [int(labels[i]) for i in np.where(zero_like_mask)[0]]

    if n_zero_like > 0:
        weakest_idx = np.argsort(absmax)[: min(5, N)]
        weak_report = ", ".join(
            f"{int(labels[i])} (max|x|={absmax[i]:.2e}, vox={roi_counts[int(labels[i])]})"
            for i in weakest_idx)
        print(f"    ↪︎ Weakest ROIs: {weak_report}")

    # Additional CSV-oriented diagnostics:
    labels_sorted     = sorted(labels.tolist())
    labels_contiguous = (labels_sorted == list(range(int(labels_sorted[0]), int(labels_sorted[0]) + N)))
    labels_from_one   = (labels_sorted == list(range(1, N + 1)))

    mean_naninf = (not np.isfinite(mean_ts).all())
    norm_naninf = (not np.isfinite(z_ts).all())
    mean_std    = np.std(mean_ts, axis=0, ddof=1)
    norm_std    = np.std(z_ts,    axis=0, ddof=1)
    mean_const  = int((mean_std < 1e-12).sum())
    norm_const  = int((norm_std < 1e-12).sum())

    norm_mu_abs  = float(np.mean(np.abs(np.mean(z_ts, axis=0))))
    norm_std_avg = float(np.mean(norm_std))

    print(f"  --> [REPORT]  {prefix}: total ROIs={N} | empty-by-count={len(empty_by_count)} "
          f"| effectively-zero={n_zero_like}")
    print(f"  --> [REPORT+] {prefix}: labels_contiguous={labels_contiguous}"
          f"{' (1..N)' if labels_from_one else ''} | "
          f"MEAN: NaN/Inf={mean_naninf}, const_cols={mean_const} | "
          f"NORM: NaN/Inf={norm_naninf}, const_cols={norm_const}, "
          f"avg|μ|={norm_mu_abs:.4f}, avgσ={norm_std_avg:.4f}")

    # Build row-wise (ROI) DataFrames: time columns t_0001..t_T:
    t_cols    = [f"t_{i:04d}" for i in range(1, T + 1)]
    roi_ids   = [int(x) for x in labels.tolist()]
    vox_cts   = [int(roi_counts[int(l)]) for l in labels]

    # MEAN: (T x N) → (N x T):
    mean_df = pd.DataFrame(
        np.transpose(mean_ts),  # <-- (N x T)
        index=roi_ids,
        columns=t_cols
    ).reset_index().rename(columns={"index": "ROI"})
    mean_df.insert(1, "num_voxels", vox_cts)

    # NORM: (T x N) → (N x T):
    norm_df = pd.DataFrame(
        np.transpose(z_ts),     # <-- (N x T)
        index=roi_ids,
        columns=t_cols
    ).reset_index().rename(columns={"index": "ROI"})
    norm_df.insert(1, "num_voxels", vox_cts)

    # Select which time-series to save based on the 'TIMESERIES_TYPE' config setting:
    if TIMESERIES_TYPE == "mean":
        out_df = mean_df
    elif TIMESERIES_TYPE == "norm":
        out_df = norm_df
    else:
        # This should never happen due to earlier config parameter validation, but keep a guard:
        raise RuntimeError(f"Unexpected TIMESERIES_TYPE at save-time: {TIMESERIES_TYPE!r}")

    # Save CSV:
    out_df.to_csv(csv_out, index=False, float_format="%.10g")

    # Create JSON sidecar with full atlas + processing metadata:
    meta = {
        "prefix": prefix,
        "n_timepoints": int(T),
        "n_rois": int(N),
        "atlas": {
            "family": ATLAS_FAMILY,
            "tag": ATLAS_TAG,
            "n_rois_requested": ATLAS_N_ROIS,
            "network_scale": int(ATLAS_NETWORK_SCALE) if ATLAS_NETWORK_SCALE is not None else None,
            "epi_filename": ATLAS_EPI_FILENAME},
        "processing": {
            "apply_gm_clip": bool(APPLY_GM_CLIP),
            "eps_zero_series": float(EPS_ZERO_SERIES),
            "timeseries_type": TIMESERIES_TYPE},
        "time_axis": {
            "indexing": "1-based post-trim",
            "columns_pattern": f"t_0001..t_{T:04d}"},
        "roi_labels": [int(x) for x in labels.tolist()],
        "roi_voxel_counts_after_mask": {str(k): int(v) for k, v in roi_counts.items()},
        "n_empty_rois_by_count": int(len(empty_by_count)),
        "n_effectively_zero_series": int(n_zero_like),
        "effectively_zero_labels": [int(x) for x in zero_like_labels],
        "csv_diagnostics": {
            "labels_contiguous": bool(labels_contiguous),
            "labels_from_one": bool(labels_from_one),
            "mean_nan_or_inf": bool(mean_naninf),
            "norm_nan_or_inf": bool(norm_naninf),
            "mean_const_cols": int(mean_const),
            "norm_const_cols": int(norm_const),
            "norm_avg_abs_mu": float(norm_mu_abs),
            "norm_avg_std": float(norm_std_avg)},
        "inputs": {
            "fmri_path": str(f_path),
            "atlas_path": str(a_path),
            "brainmask_path": str(bm_p),
            "gm_mask_path": str(gm_p),},
        "outputs": {
            "timeseries_type": TIMESERIES_TYPE,
            "csv": str(csv_out),
            "json_sidecar": str(json_sidecar)}}

    with open(json_sidecar, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"[OK] {prefix}: {TIMESERIES_TYPE.upper()} → {csv_out.name} | atlas_tag={ATLAS_TAG}")